In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# Daily

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS = np.arange(
    0.1,
    1.0,
    0.1,
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_daily"
)


# ============================================================
# LOAD TEST X / Y
# ============================================================

def load_test_xy(test_path):

    df = pd.read_csv(
        test_path
    )

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[
        x_cols
    ].to_numpy(
        dtype=float
    )

    Y = df[
        y_cols
    ].to_numpy(
        dtype=float
    )

    return X, Y


# ============================================================
# EMPIRICAL QUANTILE FORECAST
# ============================================================

def empirical_quantile_forecast(
    X,
    quantile_levels=QUANTILE_LEVELS,
):
    """
    For each forecast origin:

        1. Calculate empirical q0.1,...,q0.9
           from the 250-point context only.

        2. Use the same empirical marginal distribution
           for all 20 forecast horizons.

    Returns:
        shape (N, H, Q)

        N = number of forecast origins
        H = 20 horizons
        Q = 9 quantile levels
    """

    X = np.asarray(
        X,
        dtype=float,
    )

    # np.quantile gives:
    #
    # (Q, N)
    #
    empirical_q = np.quantile(
        X,
        quantile_levels,
        axis=1,
    )

    # Convert to:
    #
    # (N, Q)
    #
    empirical_q = empirical_q.T

    # Repeat across horizon:
    #
    # (N, 1, Q)
    #       ->
    # (N, H, Q)
    #
    forecasts = np.repeat(
        empirical_q[:, None, :],
        HORIZON,
        axis=1,
    )

    return forecasts


# ============================================================
# qCRPS
# ============================================================

def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS,
):
    """
    Same 9-quantile CRPS approximation used for the models.

    y:
        (N, H)

    quantile_forecasts:
        (N, H, Q)

    returns:
        (N, H)
    """

    y = np.asarray(
        y,
        dtype=float,
    )

    qf = np.asarray(
        quantile_forecasts,
        dtype=float,
    )

    levels = np.asarray(
        quantile_levels,
        dtype=float,
    )

    expected_shape = (
        y.shape[0],
        y.shape[1],
        len(levels),
    )

    if qf.shape != expected_shape:
        raise ValueError(
            f"Quantile forecast shape "
            f"{qf.shape} does not match "
            f"expected {expected_shape}."
        )

    error = (
        y[..., None]
        - qf
    )

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error,
    )

    crps = (
        2.0
        * np.mean(
            pinball,
            axis=-1,
        )
    )

    return crps


# ============================================================
# RUN ALL DATASETS
# ============================================================

summary_rows = []
horizon_rows = []


for asset_class, asset_name in ASSET_ORDER.items():

    print()
    print("=" * 70)
    print(f"ASSET CLASS: {asset_name}")
    print("=" * 70)

    test_asset_dir = (
        DATA_DIR
        / asset_class
    )

    test_files = sorted(
        test_asset_dir.glob(
            "test_*.csv"
        )
    )

    for test_path in test_files:

        try:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            # ------------------------------------------------
            # Test data
            # ------------------------------------------------

            X, Y = load_test_xy(
                test_path
            )

            # ------------------------------------------------
            # Empirical 9-quantile forecast
            # ------------------------------------------------

            quantiles = (
                empirical_quantile_forecast(
                    X
                )
            )

            # ------------------------------------------------
            # qCRPS
            # ------------------------------------------------

            crps = quantile_crps(
                y=Y,
                quantile_forecasts=quantiles,
                quantile_levels=QUANTILE_LEVELS,
            )

            crps_by_horizon = np.mean(
                crps,
                axis=0,
            )

            mean_qcrps = float(
                np.mean(crps)
            )

            # ------------------------------------------------
            # Summary
            # ------------------------------------------------

            summary_rows.append({
                "model": "Empirical Quantiles",
                "asset_class": asset_name,
                "dataset": dataset_name,
                "mean_qcrps": mean_qcrps,
                "n_test_origins": len(Y),
            })

            # ------------------------------------------------
            # Horizon results
            # ------------------------------------------------

            for h in range(HORIZON):

                horizon_rows.append({
                    "model": "Empirical Quantiles",
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "horizon": h + 1,
                    "mean_qcrps": (
                        crps_by_horizon[h]
                    ),
                })

            print(
                f"  {dataset_name}: "
                f"qCRPS={mean_qcrps:.8f}"
            )

        except Exception as exc:

            print(
                f"  ERROR {test_path.name}: "
                f"{exc}"
            )


# ============================================================
# DATAFRAMES
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

horizon_df = pd.DataFrame(
    horizon_rows
)


# ============================================================
# ASSET-CLASS SUMMARY
#
# Equal dataset weighting
# ============================================================

asset_summary = (
    summary_df
    .groupby(
        "asset_class",
        as_index=False,
    )
    .agg(
        mean_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        median_qcrps=(
            "mean_qcrps",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# OVERALL SUMMARY
#
# Equal weighting across datasets
# ============================================================

overall_summary = pd.DataFrame({
    "model": [
        "Empirical Quantiles"
    ],
    "mean_qcrps": [
        summary_df[
            "mean_qcrps"
        ].mean()
    ],
    "median_qcrps": [
        summary_df[
            "mean_qcrps"
        ].median()
    ],
    "n_datasets": [
        len(summary_df)
    ],
})


# ============================================================
# SAVE
# ============================================================

summary_path = (
    BASE_DIR
    / "empirical_quantile_crps_daily.csv"
)

horizon_path = (
    BASE_DIR
    / "empirical_quantile_crps_by_horizon_daily.csv"
)

asset_path = (
    BASE_DIR
    / "empirical_quantile_crps_asset_class_daily.csv"
)

overall_path = (
    BASE_DIR
    / "empirical_quantile_crps_overall_daily.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)

horizon_df.to_csv(
    horizon_path,
    index=False,
)

asset_summary.to_csv(
    asset_path,
    index=False,
)

overall_summary.to_csv(
    overall_path,
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 80)
print("EMPIRICAL QUANTILE BASELINE — DAILY")
print("=" * 80)

print()
print("Overall:")
print(
    overall_summary
    .round(6)
    .to_string(index=False)
)

print()
print("By asset class:")
print(
    asset_summary
    .round(6)
    .to_string(index=False)
)

print()
print(f"Datasets: {len(summary_df)}")

print()
print("Saved:")
print(summary_path)
print(horizon_path)
print(asset_path)
print(overall_path)

# Weekly

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

LOOKBACK = 250
HORIZON = 20

QUANTILE_LEVELS = np.arange(
    0.1,
    1.0,
    0.1,
)

ASSET_ORDER = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}


# ------------------------------------------------------------
# DAILY
# ------------------------------------------------------------

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

DATA_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_log_return"
    r"\arima_split_datasets\output_weekly"
)


# ============================================================
# LOAD TEST X / Y
# ============================================================

def load_test_xy(test_path):

    df = pd.read_csv(
        test_path
    )

    x_cols = [
        f"X_{i}"
        for i in range(LOOKBACK)
    ]

    y_cols = [
        f"Y_{i}"
        for i in range(HORIZON)
    ]

    X = df[
        x_cols
    ].to_numpy(
        dtype=float
    )

    Y = df[
        y_cols
    ].to_numpy(
        dtype=float
    )

    return X, Y


# ============================================================
# EMPIRICAL QUANTILE FORECAST
# ============================================================

def empirical_quantile_forecast(
    X,
    quantile_levels=QUANTILE_LEVELS,
):
    """
    For each forecast origin:

        1. Calculate empirical q0.1,...,q0.9
           from the 250-point context only.

        2. Use the same empirical marginal distribution
           for all 20 forecast horizons.

    Returns:
        shape (N, H, Q)

        N = number of forecast origins
        H = 20 horizons
        Q = 9 quantile levels
    """

    X = np.asarray(
        X,
        dtype=float,
    )

    # np.quantile gives:
    #
    # (Q, N)
    #
    empirical_q = np.quantile(
        X,
        quantile_levels,
        axis=1,
    )

    # Convert to:
    #
    # (N, Q)
    #
    empirical_q = empirical_q.T

    # Repeat across horizon:
    #
    # (N, 1, Q)
    #       ->
    # (N, H, Q)
    #
    forecasts = np.repeat(
        empirical_q[:, None, :],
        HORIZON,
        axis=1,
    )

    return forecasts


# ============================================================
# qCRPS
# ============================================================

def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS,
):
    """
    Same 9-quantile CRPS approximation used for the models.

    y:
        (N, H)

    quantile_forecasts:
        (N, H, Q)

    returns:
        (N, H)
    """

    y = np.asarray(
        y,
        dtype=float,
    )

    qf = np.asarray(
        quantile_forecasts,
        dtype=float,
    )

    levels = np.asarray(
        quantile_levels,
        dtype=float,
    )

    expected_shape = (
        y.shape[0],
        y.shape[1],
        len(levels),
    )

    if qf.shape != expected_shape:
        raise ValueError(
            f"Quantile forecast shape "
            f"{qf.shape} does not match "
            f"expected {expected_shape}."
        )

    error = (
        y[..., None]
        - qf
    )

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error,
    )

    crps = (
        2.0
        * np.mean(
            pinball,
            axis=-1,
        )
    )

    return crps


# ============================================================
# RUN ALL DATASETS
# ============================================================

summary_rows = []
horizon_rows = []


for asset_class, asset_name in ASSET_ORDER.items():

    print()
    print("=" * 70)
    print(f"ASSET CLASS: {asset_name}")
    print("=" * 70)

    test_asset_dir = (
        DATA_DIR
        / asset_class
    )

    test_files = sorted(
        test_asset_dir.glob(
            "test_*.csv"
        )
    )

    for test_path in test_files:

        try:

            dataset_name = (
                test_path.stem
                .removeprefix("test_")
            )

            # ------------------------------------------------
            # Test data
            # ------------------------------------------------

            X, Y = load_test_xy(
                test_path
            )

            # ------------------------------------------------
            # Empirical 9-quantile forecast
            # ------------------------------------------------

            quantiles = (
                empirical_quantile_forecast(
                    X
                )
            )

            # ------------------------------------------------
            # qCRPS
            # ------------------------------------------------

            crps = quantile_crps(
                y=Y,
                quantile_forecasts=quantiles,
                quantile_levels=QUANTILE_LEVELS,
            )

            crps_by_horizon = np.mean(
                crps,
                axis=0,
            )

            mean_qcrps = float(
                np.mean(crps)
            )

            # ------------------------------------------------
            # Summary
            # ------------------------------------------------

            summary_rows.append({
                "model": "Empirical Quantiles",
                "asset_class": asset_name,
                "dataset": dataset_name,
                "mean_qcrps": mean_qcrps,
                "n_test_origins": len(Y),
            })

            # ------------------------------------------------
            # Horizon results
            # ------------------------------------------------

            for h in range(HORIZON):

                horizon_rows.append({
                    "model": "Empirical Quantiles",
                    "asset_class": asset_name,
                    "dataset": dataset_name,
                    "horizon": h + 1,
                    "mean_qcrps": (
                        crps_by_horizon[h]
                    ),
                })

            print(
                f"  {dataset_name}: "
                f"qCRPS={mean_qcrps:.8f}"
            )

        except Exception as exc:

            print(
                f"  ERROR {test_path.name}: "
                f"{exc}"
            )


# ============================================================
# DATAFRAMES
# ============================================================

summary_df = pd.DataFrame(
    summary_rows
)

horizon_df = pd.DataFrame(
    horizon_rows
)


# ============================================================
# ASSET-CLASS SUMMARY
#
# Equal dataset weighting
# ============================================================

asset_summary = (
    summary_df
    .groupby(
        "asset_class",
        as_index=False,
    )
    .agg(
        mean_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        median_qcrps=(
            "mean_qcrps",
            "median",
        ),
        n_datasets=(
            "dataset",
            "count",
        ),
    )
)


# ============================================================
# OVERALL SUMMARY
#
# Equal weighting across datasets
# ============================================================

overall_summary = pd.DataFrame({
    "model": [
        "Empirical Quantiles"
    ],
    "mean_qcrps": [
        summary_df[
            "mean_qcrps"
        ].mean()
    ],
    "median_qcrps": [
        summary_df[
            "mean_qcrps"
        ].median()
    ],
    "n_datasets": [
        len(summary_df)
    ],
})


# ============================================================
# SAVE
# ============================================================

summary_path = (
    BASE_DIR
    / "empirical_quantile_crps_weekly.csv"
)

horizon_path = (
    BASE_DIR
    / "empirical_quantile_crps_by_horizon_weekly.csv"
)

asset_path = (
    BASE_DIR
    / "empirical_quantile_crps_asset_class_weekly.csv"
)

overall_path = (
    BASE_DIR
    / "empirical_quantile_crps_overall_weekly.csv"
)


summary_df.to_csv(
    summary_path,
    index=False,
)

horizon_df.to_csv(
    horizon_path,
    index=False,
)

asset_summary.to_csv(
    asset_path,
    index=False,
)

overall_summary.to_csv(
    overall_path,
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 80)
print("EMPIRICAL QUANTILE BASELINE — WEEKLY")
print("=" * 80)

print()
print("Overall:")
print(
    overall_summary
    .round(6)
    .to_string(index=False)
)

print()
print("By asset class:")
print(
    asset_summary
    .round(6)
    .to_string(index=False)
)

print()
print(f"Datasets: {len(summary_df)}")

print()
print("Saved:")
print(summary_path)
print(horizon_path)
print(asset_path)
print(overall_path)

# Analysis

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_daily"
)

MODEL_CRPS_PATH = (
    BASE_DIR
    / "all_models_crps_by_dataset.csv"
)

EMPIRICAL_PATH = (
    BASE_DIR
    / "empirical_quantile_crps_daily.csv"
)


# ============================================================
# LOAD
# ============================================================

models = pd.read_csv(
    MODEL_CRPS_PATH
)

empirical = pd.read_csv(
    EMPIRICAL_PATH
)


# ============================================================
# CLEAN DATASET NAMES
# ============================================================

def clean_dataset_name(name):
    """
    Convert supervised-file dataset names such as

    AUD_USD Historical Data_no_preprocess_
    lookback-250_horizon-20_slide-1

    back to

    AUD_USD Historical Data
    """

    name = str(name)

    # Remove test/train prefix if present
    name = name.removeprefix("test_")
    name = name.removeprefix("train_")

    # Remove supervised-dataset suffix
    name = name.replace(
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "",
    )

    return name


models["dataset_key"] = (
    models["dataset"]
    .apply(clean_dataset_name)
)

empirical["dataset_key"] = (
    empirical["dataset"]
    .apply(clean_dataset_name)
)


# ============================================================
# QUICK CHECK
# ============================================================

print("Example model names:")
print(
    models[
        [
            "asset_class",
            "dataset",
            "dataset_key",
        ]
    ]
    .head()
    .to_string(index=False)
)

print()

print("Example empirical names:")
print(
    empirical[
        [
            "asset_class",
            "dataset",
            "dataset_key",
        ]
    ]
    .head()
    .to_string(index=False)
)


# ============================================================
# PREP EMPIRICAL REFERENCE
# ============================================================

empirical_ref = empirical[
    [
        "asset_class",
        "dataset_key",
        "mean_qcrps",
    ]
].copy()

empirical_ref = empirical_ref.rename(
    columns={
        "mean_qcrps": "empirical_qcrps",
    }
)


# ============================================================
# MERGE
# ============================================================

comparison = models.merge(
    empirical_ref,
    on=[
        "asset_class",
        "dataset_key",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# CHECK THAT EVERYTHING MATCHED
# ============================================================

missing = comparison[
    comparison["empirical_qcrps"].isna()
][
    [
        "asset_class",
        "dataset",
        "dataset_key",
    ]
].drop_duplicates()

if len(missing) > 0:

    print()
    print("=" * 80)
    print("UNMATCHED DATASETS")
    print("=" * 80)

    print(
        missing.to_string(
            index=False
        )
    )

    raise ValueError(
        f"{len(missing)} datasets did not match "
        f"the empirical baseline."
    )


print()
print(
    "All model rows successfully matched "
    "to empirical baselines."
)


# ============================================================
# IMPROVEMENT VS EMPIRICAL QUANTILES
#
# qCRPS: lower is better
#
# Positive improvement = model beats empirical baseline
# ============================================================

comparison["absolute_improvement"] = (
    comparison["empirical_qcrps"]
    - comparison["mean_qcrps"]
)

comparison["relative_improvement_pct"] = (
    100
    * comparison["absolute_improvement"]
    / comparison["empirical_qcrps"]
)

comparison["beats_empirical"] = (
    comparison["mean_qcrps"]
    < comparison["empirical_qcrps"]
)


# ============================================================
# OVERALL MODEL SUMMARY
# ============================================================

overall_summary = (
    comparison
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_model_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        median_model_qcrps=(
            "mean_qcrps",
            "median",
        ),
        mean_empirical_qcrps=(
            "empirical_qcrps",
            "mean",
        ),
        mean_absolute_improvement=(
            "absolute_improvement",
            "mean",
        ),
        median_absolute_improvement=(
            "absolute_improvement",
            "median",
        ),
        mean_relative_improvement_pct=(
            "relative_improvement_pct",
            "mean",
        ),
        median_relative_improvement_pct=(
            "relative_improvement_pct",
            "median",
        ),
        n_datasets=(
            "dataset_key",
            "count",
        ),
        n_beats_empirical=(
            "beats_empirical",
            "sum",
        ),
    )
)


overall_summary["pct_beats_empirical"] = (
    100
    * overall_summary["n_beats_empirical"]
    / overall_summary["n_datasets"]
)


overall_summary = (
    overall_summary
    .sort_values(
        "mean_model_qcrps"
    )
    .reset_index(drop=True)
)


# ============================================================
# ASSET-CLASS SUMMARY
# ============================================================

asset_summary = (
    comparison
    .groupby(
        [
            "model",
            "asset_class",
        ],
        as_index=False,
    )
    .agg(
        mean_model_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        mean_empirical_qcrps=(
            "empirical_qcrps",
            "mean",
        ),
        mean_absolute_improvement=(
            "absolute_improvement",
            "mean",
        ),
        median_absolute_improvement=(
            "absolute_improvement",
            "median",
        ),
        mean_relative_improvement_pct=(
            "relative_improvement_pct",
            "mean",
        ),
        median_relative_improvement_pct=(
            "relative_improvement_pct",
            "median",
        ),
        n_datasets=(
            "dataset_key",
            "count",
        ),
        n_beats_empirical=(
            "beats_empirical",
            "sum",
        ),
    )
)


asset_summary["pct_beats_empirical"] = (
    100
    * asset_summary["n_beats_empirical"]
    / asset_summary["n_datasets"]
)


# ============================================================
# SAVE
# ============================================================

comparison.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_by_dataset_daily.csv",
    index=False,
)

overall_summary.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_summary_daily.csv",
    index=False,
)

asset_summary.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_asset_class_daily.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 110)
print("OVERALL COMPARISON VS EMPIRICAL QUANTILES")
print("=" * 110)

print(
    overall_summary
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 110)
print("ASSET-CLASS COMPARISON")
print("=" * 110)

print(
    asset_summary
    .round(4)
    .to_string(index=False)
)

In [ ]:
from pathlib import Path
import pandas as pd


# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(
    r"."
    r"\BForTFinProbability\BForTFin_results"
    r"\standardised\output_weekly"
)

MODEL_CRPS_PATH = (
    BASE_DIR
    / "all_models_crps_by_dataset.csv"
)

EMPIRICAL_PATH = (
    BASE_DIR
    / "empirical_quantile_crps_weekly.csv"
)


# ============================================================
# LOAD
# ============================================================

models = pd.read_csv(
    MODEL_CRPS_PATH
)

empirical = pd.read_csv(
    EMPIRICAL_PATH
)


# ============================================================
# CLEAN DATASET NAMES
# ============================================================

def clean_dataset_name(name):
    """
    Convert supervised-file dataset names such as

    AUD_USD Historical Data_no_preprocess_
    lookback-250_horizon-20_slide-1

    back to

    AUD_USD Historical Data
    """

    name = str(name)

    # Remove test/train prefix if present
    name = name.removeprefix("test_")
    name = name.removeprefix("train_")

    # Remove supervised-dataset suffix
    name = name.replace(
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "",
    )

    return name


models["dataset_key"] = (
    models["dataset"]
    .apply(clean_dataset_name)
)

empirical["dataset_key"] = (
    empirical["dataset"]
    .apply(clean_dataset_name)
)


# ============================================================
# QUICK CHECK
# ============================================================

print("Example model names:")
print(
    models[
        [
            "asset_class",
            "dataset",
            "dataset_key",
        ]
    ]
    .head()
    .to_string(index=False)
)

print()

print("Example empirical names:")
print(
    empirical[
        [
            "asset_class",
            "dataset",
            "dataset_key",
        ]
    ]
    .head()
    .to_string(index=False)
)


# ============================================================
# PREP EMPIRICAL REFERENCE
# ============================================================

empirical_ref = empirical[
    [
        "asset_class",
        "dataset_key",
        "mean_qcrps",
    ]
].copy()

empirical_ref = empirical_ref.rename(
    columns={
        "mean_qcrps": "empirical_qcrps",
    }
)


# ============================================================
# MERGE
# ============================================================

comparison = models.merge(
    empirical_ref,
    on=[
        "asset_class",
        "dataset_key",
    ],
    how="left",
    validate="many_to_one",
)


# ============================================================
# CHECK THAT EVERYTHING MATCHED
# ============================================================

missing = comparison[
    comparison["empirical_qcrps"].isna()
][
    [
        "asset_class",
        "dataset",
        "dataset_key",
    ]
].drop_duplicates()

if len(missing) > 0:

    print()
    print("=" * 80)
    print("UNMATCHED DATASETS")
    print("=" * 80)

    print(
        missing.to_string(
            index=False
        )
    )

    raise ValueError(
        f"{len(missing)} datasets did not match "
        f"the empirical baseline."
    )


print()
print(
    "All model rows successfully matched "
    "to empirical baselines."
)


# ============================================================
# IMPROVEMENT VS EMPIRICAL QUANTILES
#
# qCRPS: lower is better
#
# Positive improvement = model beats empirical baseline
# ============================================================

comparison["absolute_improvement"] = (
    comparison["empirical_qcrps"]
    - comparison["mean_qcrps"]
)

comparison["relative_improvement_pct"] = (
    100
    * comparison["absolute_improvement"]
    / comparison["empirical_qcrps"]
)

comparison["beats_empirical"] = (
    comparison["mean_qcrps"]
    < comparison["empirical_qcrps"]
)


# ============================================================
# OVERALL MODEL SUMMARY
# ============================================================

overall_summary = (
    comparison
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        mean_model_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        median_model_qcrps=(
            "mean_qcrps",
            "median",
        ),
        mean_empirical_qcrps=(
            "empirical_qcrps",
            "mean",
        ),
        mean_absolute_improvement=(
            "absolute_improvement",
            "mean",
        ),
        median_absolute_improvement=(
            "absolute_improvement",
            "median",
        ),
        mean_relative_improvement_pct=(
            "relative_improvement_pct",
            "mean",
        ),
        median_relative_improvement_pct=(
            "relative_improvement_pct",
            "median",
        ),
        n_datasets=(
            "dataset_key",
            "count",
        ),
        n_beats_empirical=(
            "beats_empirical",
            "sum",
        ),
    )
)


overall_summary["pct_beats_empirical"] = (
    100
    * overall_summary["n_beats_empirical"]
    / overall_summary["n_datasets"]
)


overall_summary = (
    overall_summary
    .sort_values(
        "mean_model_qcrps"
    )
    .reset_index(drop=True)
)


# ============================================================
# ASSET-CLASS SUMMARY
# ============================================================

asset_summary = (
    comparison
    .groupby(
        [
            "model",
            "asset_class",
        ],
        as_index=False,
    )
    .agg(
        mean_model_qcrps=(
            "mean_qcrps",
            "mean",
        ),
        mean_empirical_qcrps=(
            "empirical_qcrps",
            "mean",
        ),
        mean_absolute_improvement=(
            "absolute_improvement",
            "mean",
        ),
        median_absolute_improvement=(
            "absolute_improvement",
            "median",
        ),
        mean_relative_improvement_pct=(
            "relative_improvement_pct",
            "mean",
        ),
        median_relative_improvement_pct=(
            "relative_improvement_pct",
            "median",
        ),
        n_datasets=(
            "dataset_key",
            "count",
        ),
        n_beats_empirical=(
            "beats_empirical",
            "sum",
        ),
    )
)


asset_summary["pct_beats_empirical"] = (
    100
    * asset_summary["n_beats_empirical"]
    / asset_summary["n_datasets"]
)


# ============================================================
# SAVE
# ============================================================

comparison.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_by_dataset_weekly.csv",
    index=False,
)

overall_summary.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_summary_weekly.csv",
    index=False,
)

asset_summary.to_csv(
    BASE_DIR
    / "empirical_crps_comparison_asset_class_weekly.csv",
    index=False,
)


# ============================================================
# PRINT
# ============================================================

print()
print("=" * 110)
print("OVERALL COMPARISON VS EMPIRICAL QUANTILES")
print("=" * 110)

print(
    overall_summary
    .round(4)
    .to_string(index=False)
)


print()
print("=" * 110)
print("ASSET-CLASS COMPARISON")
print("=" * 110)

print(
    asset_summary
    .round(4)
    .to_string(index=False)
)